# Full-bill LLooM concept pipeline (8,317 bills)

Scales LLooM concept discovery to the full vote-linked corpus, checkpointing every
stage to **parquet** so you can stop, resume, and change clustering parameters
without repaying for Gemini embeddings. Same spirit as `lloom_experiment_v2.ipynb`.

The **interesting pipeline flow is here, in the cells**: the LLooM `distill_filter` /
`distill_summarize` / `synthesize` / `review` calls, the Gemini embedding loop, and
the seeded UMAP/HDBSCAN clustering are all explicit. `lawsofisrael.concepts` (`cp`)
only holds mechanical plumbing: parquet checkpoints, the document-selection policy,
the extraction/chunking wrappers, and the final bill-level aggregation.

Rules: **no KNN**, **no scoring**. Clusters are the result; concept labels are
semantic descriptions of clusters. Every paid call is content-keyed, so a rerun
skips rows already done.

## 0. Setup, paths, parameters

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys
import asyncio
import pandas as pd

ROOT = Path.cwd().resolve().parent
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "src"))

from config import (
    MODEL_CONFIG, MAX_OUTPUT_TOKENS,
    GEMINI_EMBEDDING_MODEL, GEMINI_EMBEDDING_DIM,
    GEMINI_EMBEDDING_BASE_URL, GEMINI_EMBEDDING_API_KEY,
)
from lawsofisrael import concepts as cp, prompts_v2
from lawsofisrael import diagnostics_v2 as diag
from lawsofisrael.lloom import build_chat_model, build_gemini_embed_model

# text_lloom primitives — called directly below so the flow is visible.
from text_lloom.concept_induction import (
    distill_filter, distill_summarize, synthesize, review,
)
from text_lloom.prompts import filter_prompt

# --- Inputs (from the vote-scrape pipeline) --------------------------------
PREPARED = ROOT / "dataset" / "knesset_votes" / "prepared"
BILLS_PATH = PREPARED / "dim_bills.parquet"
DOCUMENTS_PATH = PREPARED / "dim_bill_documents.parquet"

# --- Run outputs (parquet checkpoints) -------------------------------------
RUN = ROOT / "dataset" / "bill_concepts" / "runs" / "full-v1"
PDF_STORE = ROOT / "dataset" / "bill_concepts" / "raw" / "pdfs" / "sha256"
RUN.mkdir(parents=True, exist_ok=True)
def path(name):
    return RUN / f"{name}.parquet"

# --- Parameters ------------------------------------------------------------
N_BILLS = 200            # None = all 8,317; int = sample for iteration
CHUNK_TOKENS = 3000
FILTER_N_QUOTES = 5
SUMMARY_N_BULLETS = 7
SYNTH_N_CONCEPTS = 1
SEED = 42
LLM_BATCH = 25
EMBED_DIM = GEMINI_EMBEDDING_DIM or 3072
EMBED_BATCH = 64
EMBED_INSTRUCTION = diag.DEFAULT_EMBED_INSTRUCTION
USE_DOMAIN_CONTEXT = True

# Seeded UMAP/HDBSCAN candidates (local; cheap to re-run from saved vectors).
CLUSTER_CANDIDATES = {
    "A": dict(hdbscan_min_cluster_size=3, hdbscan_cluster_selection_method="leaf"),
    "B": dict(hdbscan_min_cluster_size=5, hdbscan_cluster_selection_method="eom"),
    "C": dict(hdbscan_min_cluster_size=5, hdbscan_cluster_selection_method="leaf"),
    "D": dict(hdbscan_min_cluster_size=7, hdbscan_cluster_selection_method="eom"),
    "E": dict(hdbscan_min_cluster_size=4, hdbscan_cluster_selection_method="leaf"),
}
CHOSEN_CANDIDATE = "E"

SUMMARIZE_INFO = prompts_v2.distill_summarize_prompt_info()
SYNTH_INFO = prompts_v2.synthesize_prompt_info()
print("Run dir:", RUN)
print("Summarize prompt:", SUMMARIZE_INFO, "| Synthesize prompt:", SYNTH_INFO)

## 1. Load inputs and select one document per bill

Selection policy (prefer official published-law PDF for passed bills, else ranked
fallback; keep bills with no usable PDF) lives in `cp.select_documents` — it is a
long deterministic policy, not pipeline logic.

In [ ]:
bills = pd.read_parquet(BILLS_PATH)
if N_BILLS:
    bills = bills.head(N_BILLS).copy()
documents = pd.read_parquet(DOCUMENTS_PATH)

selection = cp.select_documents(bills, documents)
cp.write_parquet(selection, path("document_selection"))
print(selection["selection_state"].value_counts().to_dict())
selection.head()

## 2. Download selected PDFs (local, content-addressed) — network

In [ ]:
pdf_manifest = cp.download_pdfs(selection, PDF_STORE, prior=cp.read_parquet(path("pdf_manifest")))
cp.write_parquet(pdf_manifest, path("pdf_manifest"))
print(pdf_manifest["download_state"].value_counts().to_dict())

## 3. Extract PDFs and chunk (local, provenance-preserving)

In [ ]:
ext_docs, ext_blocks = cp.extract_pdfs(
    pdf_manifest, bills,
    prior_documents=cp.read_parquet(path("extraction_documents")),
    prior_blocks=cp.read_parquet(path("extraction_blocks")),
)
cp.write_parquet(ext_docs, path("extraction_documents"))
cp.write_parquet(ext_blocks, path("extraction_blocks"))

chunks = cp.build_chunks(ext_docs, ext_blocks, bills, chunk_tokens=CHUNK_TOKENS)
cp.write_parquet(chunks, path("chunks"))
print("extracted:", (ext_docs["extraction_state"] == "success").sum(),
      "blocks:", len(ext_blocks), "chunks:", len(chunks))
chunks[["chunk_id", "bill_id", "bill_title", "page_no", "tokens"]].head()

## 4. Distill — LLooM quote filter + condition-faithful summaries (paid)

This is LLooM's `distill_filter` then `distill_summarize`, called directly. The only
thing wrapped around them is a content-key cache: a chunk already summarized in
`bullets.parquet` is skipped, and each batch is written before the next so an
interrupted run keeps its progress. We build the chat model exactly like v2.

In [ ]:
chat_model = build_chat_model(
    model_config=MODEL_CONFIG, base_url=MODEL_CONFIG["base_url"],
    api_key=MODEL_CONFIG["api_key"], max_output_tokens=MAX_OUTPUT_TOKENS,
)

bullets = cp.read_parquet(path("bullets"))
done_chunks = set(bullets["chunk_id"].astype(str)) if bullets is not None else set()
todo = chunks[~chunks["chunk_id"].astype(str).isin(done_chunks)]
print(f"{len(todo)} chunks to distill ({len(done_chunks)} already have bullets)")

bullet_rows = [] if bullets is None else bullets.to_dict("records")
for start in range(0, len(todo), LLM_BATCH):
    batch = todo.iloc[start:start + LLM_BATCH]

    # 1) LLooM quote filter (default prompt), batch of chunks -> quotes per chunk.
    filtered = await distill_filter(
        batch[["chunk_id", "text"]].rename(columns={"text": "text"}),
        doc_col="text", doc_id_col="chunk_id", model=chat_model,
        n_quotes=FILTER_N_QUOTES, prompt_template=filter_prompt, seed=SEED,
    )
    if filtered.empty:
        print(f"  batch {start}: no quotes parsed, skipping"); continue

    # 2) v2 condition-faithful summaries over the filtered quotes -> bullets.
    summarized = await distill_summarize(
        filtered, doc_col="text", doc_id_col="chunk_id", model=chat_model,
        n_bullets=SUMMARY_N_BULLETS, prompt_template=prompts_v2.DISTILL_SUMMARIZE_PROMPT,
        seed=SEED,
    )

    # 3) explode to one row per bullet with a stable id + prompt provenance.
    for cid, group in summarized.groupby(summarized["chunk_id"].astype(str), sort=False):
        for ordinal, text in enumerate(b for b in group["text"] if str(b).strip()):
            bullet_rows.append({
                "bullet_row_id": f"{cid}#b{ordinal:03d}", "chunk_id": cid,
                "bullet_ordinal": ordinal, "bullet": text,
                "summarize_prompt_version": SUMMARIZE_INFO["summarize_prompt_version"],
                "summarize_prompt_hash": SUMMARIZE_INFO["summarize_prompt_hash"],
            })
    bullets = pd.DataFrame(bullet_rows).drop_duplicates("bullet_row_id", keep="last")
    cp.write_parquet(bullets, path("bullets"))
    print(f"  chunks {min(start + LLM_BATCH, len(todo))}/{len(todo)} -> bullets={len(bullets)}")

print("total bullets:", len(bullets),
      "| chunks with >=1 bullet:", bullets["chunk_id"].nunique(), "of", len(chunks))

## 5. Embed bullets with Gemini (paid)

Compose the exact embedding input (instruction + bill title/heading + bullet) with
`diag.build_embedding_input`, key each vector by input+model+dim, and only call
Gemini for bullets not already embedded. Re-clustering never re-embeds.

In [ ]:
embed_model = build_gemini_embed_model(
    base_url=GEMINI_EMBEDDING_BASE_URL, api_key=GEMINI_EMBEDDING_API_KEY,
    model_name=GEMINI_EMBEDDING_MODEL, dimensions=EMBED_DIM, batch_size=EMBED_BATCH,
)
context_by_chunk = chunks.set_index("chunk_id")[["bill_title", "heading_path"]].to_dict("index")

def embed_input(row):
    ctx = context_by_chunk.get(row["chunk_id"]) if USE_DOMAIN_CONTEXT else None
    # make_chunk_lookup-style context: build_embedding_input reads name/heading_path
    ctx = {"name": ctx["bill_title"], "heading_path": ctx["heading_path"]} if ctx else None
    return diag.build_embedding_input(row["bullet"], instruction=EMBED_INSTRUCTION, context=ctx)

emb_inputs = bullets.copy()
emb_inputs["embedding_input"] = emb_inputs.apply(embed_input, axis=1)
emb_inputs["embedding_request_key"] = emb_inputs["embedding_input"].map(
    lambda t: cp.content_hash({"text": t, "model": GEMINI_EMBEDDING_MODEL, "dim": EMBED_DIM}))
cp.write_parquet(emb_inputs[["embedding_request_key", "bullet_row_id", "chunk_id", "embedding_input"]],
                 path("embedding_inputs"))

embeddings = cp.read_parquet(path("embeddings"))
done_keys = set(embeddings["embedding_request_key"]) if embeddings is not None else set()
todo = emb_inputs[~emb_inputs["embedding_request_key"].isin(done_keys)]
print(f"{len(todo)} bullets to embed ({len(done_keys)} cached)")

rows = [] if embeddings is None else embeddings.to_dict("records")
for start in range(0, len(todo), EMBED_BATCH):
    batch = todo.iloc[start:start + EMBED_BATCH]
    vectors, _ = embed_model.fn(embed_model, batch["embedding_input"].tolist())  # Gemini call
    for (_, item), vector in zip(batch.iterrows(), vectors):
        rows.append({"embedding_request_key": item["embedding_request_key"],
                     "bullet_row_id": item["bullet_row_id"], "vector": list(map(float, vector))})
    embeddings = pd.DataFrame(rows).drop_duplicates("embedding_request_key", keep="last")
    cp.write_parquet(embeddings, path("embeddings"))
    print(f"  embedded {min(start + EMBED_BATCH, len(todo))}/{len(todo)}")
print("total embeddings:", len(embeddings))

## 6. Cluster — seeded UMAP + HDBSCAN (local)

`diag.run_clustering` (seeded, reproducible) over the saved Gemini vectors, once per
candidate. Inspect the summary (noise, cross-bill vs single-bill), then set
`CHOSEN_CANDIDATE` in cell 0. No API calls here, so re-run freely.

In [ ]:
import numpy as np

aligned = (emb_inputs[["bullet_row_id", "embedding_request_key", "chunk_id"]]
           .merge(embeddings, on=["embedding_request_key", "bullet_row_id"], how="inner")
           .sort_values("bullet_row_id", kind="stable").reset_index(drop=True))
ids = aligned["bullet_row_id"].tolist()
vectors = np.vstack(aligned["vector"].map(lambda v: np.asarray(v, dtype=np.float32)))
bill_of = {bid: cid.split(":", 1)[0] for bid, cid in zip(aligned["bullet_row_id"], aligned["chunk_id"].astype(str))}
data_hash = cp.content_hash(aligned["embedding_request_key"].tolist())[:12]

assign_rows, summary_rows, results = [], [], {}
for key, params in CLUSTER_CANDIDATES.items():
    cfg = diag.ClusterConfig(umap_n_neighbors=15, umap_n_components=5, umap_min_dist=0.0,
                             random_state=SEED, **params)
    res = diag.run_clustering(ids, [""] * len(ids), vectors, cfg)
    run_id = f"{key}_{cfg.run_id()}_{data_hash}"
    results[key] = (run_id, res)
    for i, bid in enumerate(ids):
        assign_rows.append({"candidate_key": key, "cluster_run_id": run_id, "bullet_row_id": bid,
                            "cluster_id": int(res.labels[i]), "is_noise": int(res.labels[i]) == -1,
                            "membership_probability": float(res.probabilities[i]),
                            "outlier_score": float(res.outlier_scores[i])})
    for comp in res.cluster_composition():
        summary_rows.append({"candidate_key": key, "cluster_run_id": run_id, **comp})
    print(f"[{key}] {run_id}: clusters={res.n_clusters()} noise={res.noise_ratio():.0%} "
          f"cross_bill={res.n_cross_bill_clusters()} single_bill={res.n_single_bill_clusters()}")

assignments = pd.DataFrame(assign_rows)
cp.write_parquet(assignments, path("cluster_assignments"))
cp.write_parquet(pd.DataFrame(summary_rows), path("cluster_summaries"))

## 7. Synthesize concept labels for the chosen clustering — LLooM synthesize + review (paid, no scoring)

In [ ]:
run_id, _ = results[CHOSEN_CANDIDATE]
chosen = assignments[assignments["cluster_run_id"] == run_id].copy()
chosen = chosen.merge(bullets[["bullet_row_id", "bullet"]], on="bullet_row_id", how="left")
print("chosen:", run_id, "| clusters:", chosen.loc[chosen.cluster_id != -1, "cluster_id"].nunique())

concepts = cp.read_parquet(path("concepts"))
concept_rows = [] if concepts is None else concepts.to_dict("records")
done_clusters = set((concepts["cluster_run_id"] + ":" + concepts["cluster_id"].astype(str))) if concepts is not None else set()

for cluster_id, group in chosen[chosen.cluster_id != -1].groupby("cluster_id", sort=True):
    tag = f"{run_id}:{cluster_id}"
    if tag in done_clusters:
        continue
    # LLooM synthesize expects [doc_id, doc, cluster_id]; keep bullet ids as example ids.
    cluster_df = group[["bullet_row_id", "bullet"]].rename(columns={"bullet_row_id": "example_id", "bullet": "text"})
    cluster_df["cluster_id"] = int(cluster_id)
    concept_map, concept_df, _ = await synthesize(
        cluster_df, doc_col="text", doc_id_col="example_id", model=chat_model,
        cluster_id_col="cluster_id", n_concepts=SYNTH_N_CONCEPTS, pattern_phrase="unique topic",
        prompt_template=prompts_v2.SYNTHESIZE_PROMPT, seed=SEED, return_concepts_dict=True,
    )
    if concept_map:
        concept_map, concept_df = await review(concept_map, concept_df, "concept", chat_model, seed=SEED)
    for ordinal, c in enumerate(sorted(concept_map.values(), key=lambda c: (c.name, c.prompt))):
        concept_rows.append({
            "concept_id": f"concept_{cp.content_hash({'run': run_id, 'cluster': int(cluster_id), 'name': c.name, 'prompt': c.prompt})[:24]}",
            "cluster_run_id": run_id, "cluster_id": int(cluster_id), "concept_ordinal": ordinal,
            "concept_label": c.name, "concept_criterion": c.prompt,
            "representative_bullet_ids": sorted(str(v) for v in c.example_ids),
            "synthesize_prompt_version": SYNTH_INFO["synthesize_prompt_version"],
            "synthesize_prompt_hash": SYNTH_INFO["synthesize_prompt_hash"],
        })
    concepts = pd.DataFrame(concept_rows).drop_duplicates("concept_id", keep="last")
    cp.write_parquet(concepts, path("concepts"))
    print(f"  cluster {cluster_id} -> concepts={len(concepts)}")
concepts[["cluster_id", "concept_label", "concept_criterion"]].head(20)

## 8. Build query-ready bill-level marts

`cp.build_analysis` is the long pure-pandas aggregation. `bridge_bill_clusters` is
direct HDBSCAN membership per bill; `bridge_bill_concepts` inherits labels from each
bill's clusters; `bill_concepts` is one row per bill. Probability columns are
membership strengths, not scores.

In [ ]:
marts = cp.build_analysis(bills, chunks, bullets, chosen, concepts)
for name, frame in marts.items():
    cp.write_parquet(frame, path(name))
    print(f"{name}: {len(frame)} rows")

report = cp.quality_report(bills, bullets, chosen, marts)
cp.write_parquet(report, path("quality_report"))
display(report)

## 9. Trace one concept back to source

In [ ]:
bbc = marts["bridge_bill_concepts"]
if len(bbc):
    row = bbc.iloc[0]
    chunk = chunks[chunks["chunk_id"] == row["primary_evidence_chunk_id"]].iloc[0]
    print("CONCEPT: ", row["concept_label"])
    print("CRITERION:", row["concept_criterion"])
    print("RELATION:", row["relation_type"], "| cluster", row["cluster_id"])
    print("BILL:    ", row["bill_id"], "-", chunk["bill_title"])
    print("PAGE:    ", chunk["page_no"], "| pdf", chunk["pdf_sha256"])
    print("SOURCE (excerpt):", (chunk["text"] or "")[:400])
else:
    print("No concepts yet.")